In [315]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [316]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv')
data.drop(columns=['API_UserName','observations','dayofweek','is_weekend','day','month'], inplace=True)
data.head(40)

,date,indicator,seen
0,2025-01-01,146.71.50.198,1
1,2025-01-01,149.36.49.225,1
2,2025-01-01,162.142.125.242,1
3,2025-01-01,162.142.125.247,1
4,2025-01-01,162.142.125.255,1
5,2025-01-01,185.230.63.171,1
6,2025-01-01,23.26.221.12,1
7,2025-01-01,23.26.221.2,1
8,2025-01-01,23.26.221.4,1
9,2025-01-01,34.160.111.145,1


In [317]:
# Ensure the 'date' column is in datetime format
data['date'] = pd.to_datetime(data['date'])

# Keep the last 7 days in test_data
last_7_days = datetime.now() - timedelta(days=7)
# Normalize today's date (removes the time portion)
today = pd.to_datetime(datetime.now()).normalize()

# Filter test_data: only indicators seen today
test_data = data[(data['date'] == today) & (data['seen'] == 1)]
 
test_data = test_data.reset_index(drop=True)

# Use the remaining data for training
data = data[data['date'] < last_7_days]

# Reset the index of the filtered data
data = data.reset_index(drop=True)
# Display the test_data
test_data

,date,indicator,seen
0,2025-04-17,162.142.125.242,1
1,2025-04-17,162.142.125.247,1
2,2025-04-17,162.142.125.255,1
3,2025-04-17,185.230.63.171,1
4,2025-04-17,34.160.111.145,1
5,2025-04-17,104.18.32.191,1
6,2025-04-17,104.21.48.1,1
7,2025-04-17,23.205.105.180,1
8,2025-04-17,104.21.54.132,1
9,2025-04-17,68.67.179.164,1


In [318]:
# Define the cutoff date (90 days ago from today)
cutoff_date = datetime.now() - timedelta(days=120)

# Convert the date column to datetime format (if not already)
data['date'] = pd.to_datetime(data['date'])

# Filter the data to include only rows within the last 90 days
filtered_data = data[data['date'] >= cutoff_date]
filtered_data

,date,indicator,seen
0,2025-01-01,146.71.50.198,1
1,2025-01-01,149.36.49.225,1
2,2025-01-01,162.142.125.242,1
3,2025-01-01,162.142.125.247,1
4,2025-01-01,162.142.125.255,1
...,...,...,...
19795,2025-04-10,190.92.174.92,0
19796,2025-04-10,20.48.204.4,0
19797,2025-04-10,47.237.115.193,0
19798,2025-04-10,107.180.119.251,0


In [319]:
unique_indicators_count = filtered_data['indicator'].nunique()
print(f"Number of unique indicators: {unique_indicators_count}")

Number of unique indicators: 198


In [320]:
target_indicator = '104.18.32.191'
indicator_data = test_data[test_data['indicator'] == target_indicator]
print(indicator_data)


        date      indicator  seen
5 2025-04-17  104.18.32.191     1


In [321]:
indicator_data = data[data['indicator'] == '104.18.32.191']
seen_counts = indicator_data['seen'].value_counts()
print(seen_counts)



seen
0    82
1    18
Name: count, dtype: int64


In [322]:
def days_since_last_seen_all(data):
    # Filter only rows where seen == 1
    seen_data = data[data['seen'] == 1]
    
    # Sort the data by indicator and date in descending order
    seen_data = seen_data.sort_values(by=['indicator', 'date'], ascending=[True, False])
    
    # Keep only the most recent entry for each indicator
    seen_data = seen_data.drop_duplicates(subset=['indicator'], keep='first')
    
    # Calculate the difference in days from today for each entry
    today = pd.to_datetime(datetime.now()).normalize()
    seen_data['days_since_last_seen'] = (today - seen_data['date']).dt.days
    
    # Calculate the number of appearances in the last 30 days
    last_30_days = today - timedelta(days=30)
    appearances_last_30_days = data[(data['seen'] == 1) & (data['date'] >= last_30_days)]
    appearances_count_30 = appearances_last_30_days.groupby('indicator').size()
    seen_data['appearances_last_30_days'] = seen_data['indicator'].map(appearances_count_30).fillna(0).astype(int)
    
    # Calculate the number of appearances in the last 7 days
    last_7_days = today - timedelta(days=7)
    appearances_last_7_days = data[(data['seen'] == 1) & (data['date'] >= last_7_days)]
    appearances_count_7 = appearances_last_7_days.groupby('indicator').size()
    seen_data['appearances_last_7_days'] = seen_data['indicator'].map(appearances_count_7).fillna(0).astype(int)
    
    # Drop rows with NaN values
    seen_data = seen_data.dropna()
    
    return seen_data[['indicator', 'date', 'seen', 'days_since_last_seen', 'appearances_last_30_days', 'appearances_last_7_days']]

# Apply the function to the entire dataset
days_since_last_seen_all_data = days_since_last_seen_all(data)

# Display the result
days_since_last_seen_all_data

,indicator,date,seen,days_since_last_seen,appearances_last_30_days,appearances_last_7_days
17189,102.129.153.158,2025-03-28,1,20,1,0
12420,102.129.153.43,2025-03-04,1,44,0,0
13872,102.129.153.71,2025-03-12,1,36,0,0
19764,102.165.16.161,2025-04-10,1,7,3,1
14225,104.160.6.2,2025-03-13,1,35,0,0
...,...,...,...,...,...,...
4071,international.standardbank.com/,2025-01-21,1,86,0,0
19791,pub.marq.com/,2025-04-10,1,7,5,1
19147,realinvestmentadvice.com/,2025-04-07,1,10,2,0
8441,www.emergencylighting.com/,2025-02-12,1,64,0,0


In [323]:
# Compute per-indicator rolling 7-day reappearance probability
rolling_prob_list = []

# Process each indicator individually
for indicator, group in filtered_data.groupby('indicator'):
    group = group.sort_values('date').reset_index(drop=True)
    
    # Create target: was it seen 7 days later?
    group['seen_7_days_later'] = group['seen'].shift(-7)
    
    # Rolling mean to estimate probability of reappearance
    group['prob_7_day_reappearance'] = group['seen_7_days_later'].rolling(window=30, min_periods=1).mean()
    
    # Save with indicator name
    group['indicator'] = indicator
    rolling_prob_list.append(group)

# Combine all into one DataFrame
rolling_probs = pd.concat(rolling_prob_list).reset_index(drop=True)

# Show sample of results
rolling_probs[['date', 'indicator', 'seen', 'prob_7_day_reappearance']].sort_values(by='prob_7_day_reappearance', ascending=False).head(100)


,date,indicator,seen,prob_7_day_reappearance
870,2025-03-12,104.21.48.1,1,1.000000
871,2025-03-13,104.21.48.1,1,1.000000
872,2025-03-14,104.21.48.1,1,1.000000
868,2025-03-10,104.21.48.1,1,1.000000
869,2025-03-11,104.21.48.1,1,1.000000
...,...,...,...,...
876,2025-03-18,104.21.48.1,1,0.933333
879,2025-03-21,104.21.48.1,1,0.933333
842,2025-02-12,104.21.48.1,1,0.933333
5064,2025-03-06,162.142.125.242,1,0.933333


In [324]:
from prophet import Prophet

# Get unique indicators
indicators = rolling_probs['indicator'].unique()

# Store results
forecast_results = []

# Loop through each indicator
for ip in indicators:
    ip_data = rolling_probs[rolling_probs['indicator'] == ip].copy()
    
    # Prepare for Prophet
    prophet_df = ip_data[['date', 'prob_7_day_reappearance']].dropna().rename(columns={
        'date': 'ds',
        'prob_7_day_reappearance': 'y'
    })
    
    # Skip if there's not enough data
    if len(prophet_df) < 30:
        continue

    prophet_df['cap'] = 1.0
    prophet_df['floor'] = 0.0

    try:
        # Fit model
        model = Prophet(growth='logistic')
        model.fit(prophet_df)

        # Forecast 7 days ahead
        future = model.make_future_dataframe(periods=7)
        future['cap'] = 1.0
        future['floor'] = 0.0
        forecast = model.predict(future)

        # Get just the prediction for the 7th day (last day of forecast)
        day_7_forecast = forecast[['ds', 'yhat']].iloc[-1:].copy()
        day_7_forecast['indicator'] = ip
        forecast_results.append(day_7_forecast)

    
    except Exception as e:
        print(f"Skipped {ip} due to error: {e}")

# Combine results into one DataFrame
forecast_df = pd.concat(forecast_results).reset_index(drop=True)
forecast_df['yhat_percent'] = (forecast_df['yhat'] * 100).round(2).astype(str) + '%'
forecast_df


10:33:47 - cmdstanpy - INFO - Chain [1] start processing
10:33:47 - cmdstanpy - INFO - Chain [1] done processing
10:33:47 - cmdstanpy - INFO - Chain [1] start processing
10:33:47 - cmdstanpy - INFO - Chain [1] done processing
10:33:47 - cmdstanpy - INFO - Chain [1] start processing
10:33:47 - cmdstanpy - INFO - Chain [1] done processing
10:33:48 - cmdstanpy - INFO - Chain [1] start processing
10:33:48 - cmdstanpy - INFO - Chain [1] done processing
10:33:48 - cmdstanpy - INFO - Chain [1] start processing
10:33:48 - cmdstanpy - INFO - Chain [1] done processing
10:33:49 - cmdstanpy - INFO - Chain [1] start processing
10:33:49 - cmdstanpy - INFO - Chain [1] done processing
10:33:49 - cmdstanpy - INFO - Chain [1] start processing
10:33:49 - cmdstanpy - INFO - Chain [1] done processing
10:33:49 - cmdstanpy - INFO - Chain [1] start processing
10:33:50 - cmdstanpy - INFO - Chain [1] done processing
10:33:50 - cmdstanpy - INFO - Chain [1] start processing
10:33:50 - cmdstanpy - INFO - Chain [1]

,ds,yhat,indicator,yhat_percent
0,2025-04-17,0.066446,102.129.153.158,6.64%
1,2025-04-17,0.020869,102.129.153.43,2.09%
2,2025-04-17,0.043408,102.129.153.71,4.34%
3,2025-04-17,0.260356,102.165.16.161,26.04%
4,2025-04-17,0.011307,104.160.6.2,1.13%
...,...,...,...,...
193,2025-04-17,0.012002,international.standardbank.com/,1.2%
194,2025-04-17,0.389662,pub.marq.com/,38.97%
195,2025-04-17,0.154591,realinvestmentadvice.com/,15.46%
196,2025-04-17,0.032026,www.emergencylighting.com/,3.2%


In [325]:
# Extract the indicators from both dataframes
sorted_results_indicators = set(forecast_df['indicator'])
test_data_indicators = set(test_data['indicator'])

# Find matching indicators
matching_indicators = sorted_results_indicators.intersection(test_data_indicators)

# Find missing indicators in test_data
missing_in_test_data = sorted_results_indicators.difference(test_data_indicators)

# Find missing indicators in sorted_results_df
missing_in_sorted_results = test_data_indicators.difference(sorted_results_indicators)

# Display the results
print("Matching Indicators:", matching_indicators)
print("Indicators in sorted_results_df but missing in test_data:", missing_in_test_data)
print("Indicators in test_data but missing in sorted_results_df:", missing_in_sorted_results)

Matching Indicators: {'66.132.159.247', '34.160.111.145', '162.142.125.255', '104.21.48.1', '162.142.125.247', '23.205.105.180', '104.18.32.191', '162.142.125.242', '68.67.179.164', '104.21.54.132', '185.230.63.171'}
Indicators in sorted_results_df but missing in test_data: {'156.146.63.166', '143.95.44.89', '15.235.218.150', '107.180.119.251', '191.96.36.115', '216.24.216.204', '23.26.221.18', '23.26.221.26', '46.246.8.63', '46.246.8.84', '185.38.109.109', '46.246.8.29', '46.246.8.43', '74.119.239.234', '173.208.96.45', '113.96.236.11', '23.26.221.27', '149.36.49.225', '20.233.253.125', '46.246.8.122', '156.146.63.174', '70.39.115.74', '162.241.225.237', 'international.standardbank.com/', '23.26.221.17', '64.64.112.145', '146.70.204.179', '46.246.8.96', '45.142.193.141', '156.146.63.180', '46.246.8.131', '64.64.112.159', '66.132.159.244', '46.246.8.102', '102.129.153.71', '88.119.174.148', '23.26.221.6', '46.246.8.119', '149.102.250.14', '45.142.193.207', '46.246.8.9', '171.25.193.20'

In [326]:
precision = len(matching_indicators) / len(sorted_results_indicators)
recall = len(matching_indicators) / len(test_data_indicators)
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")


Precision: 0.06
Recall: 1.00
F1 Score: 0.11


In [327]:
# Filter the missing indicators from forecast_df
missing_indicators_data = forecast_df[forecast_df['indicator'].isin(missing_in_test_data)]

# Display the results
missing_indicators_data[['indicator', 'yhat_percent']]

,indicator,yhat_percent
0,102.129.153.158,6.64%
1,102.129.153.43,2.09%
2,102.129.153.71,4.34%
3,102.165.16.161,26.04%
4,104.160.6.2,1.13%
...,...,...
193,international.standardbank.com/,1.2%
194,pub.marq.com/,38.97%
195,realinvestmentadvice.com/,15.46%
196,www.emergencylighting.com/,3.2%


In [328]:
# Filter the matching indicators from forecast_df
matching_indicators_data = forecast_df[forecast_df['indicator'].isin(matching_indicators)]

# Get the list of indicators from matching_indicators_data
matching_indicators = matching_indicators_data['indicator'].unique()

# Filter the last 90 days of data for these indicators where seen = 1
seen_in_last_90_days = filtered_data[(filtered_data['indicator'].isin(matching_indicators)) & (filtered_data['seen'] == 1)]

# Get the list of indicators that have been seen
seen_indicators = seen_in_last_90_days['indicator'].unique()

# Find indicators that have not been seen in the last 90 days
not_seen_indicators = set(matching_indicators) - set(seen_indicators)

# Display the indicators not seen in the last 90 days
if not_seen_indicators:
    print("The following indicators have NOT been seen in the last 90 days:")
    print(not_seen_indicators)
else:
    print("All matching indicators have been seen in the last 90 days.")

# Exclude not_seen_indicators from the display list
display_indicators = set(matching_indicators) - not_seen_indicators

# Display the results
matching_indicators_data[matching_indicators_data['indicator'].isin(display_indicators)][['indicator', 'yhat_percent']]


The following indicators have NOT been seen in the last 90 days:
{'66.132.159.247'}


,indicator,yhat_percent
5,104.18.32.191,1.55%
8,104.21.48.1,99.75%
9,104.21.54.132,97.12%
50,162.142.125.242,80.93%
51,162.142.125.247,56.75%
52,162.142.125.255,67.3%
65,185.230.63.171,77.7%
82,23.205.105.180,33.15%
115,34.160.111.145,84.03%
176,68.67.179.164,88.95%
